# BIOSTAT 707 Checkpoint 1

**Author:** Tianyu Xu  
**NetID:** tx54

## Challenge 2012 Set A Cohort Characterization

This notebook characterizes the Challenge 2012 Set A cohort, including data wrangling, cohort characteristics, outcome summaries, and patterns of missing data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
ROOT = Path.cwd()

# Make sure the notebook is being run from the repository root
assert (ROOT / "checkpoint1.ipynb").exists(), \
    "checkpoint1.ipynb must be in the repository root."

DATA = ROOT / "data"
SET_A_DIR = DATA / "set-a"
OUTCOMES_FILE = DATA / "Outcomes-a.txt"

OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

print("Project root:", ROOT)
print("Data directory:", DATA)
print("Output directory:", OUT)

In [ ]:
# Check required input data

assert SET_A_DIR.exists(), f"Missing directory: {SET_A_DIR}"
assert OUTCOMES_FILE.exists(), f"Missing file: {OUTCOMES_FILE}"

patient_files = sorted(SET_A_DIR.glob("*.txt"))

print("Number of patient files:", len(patient_files))
print("Outcomes file exists:", OUTCOMES_FILE.exists())

assert len(patient_files) == 4000, \
    f"Expected 4000 patient files, found {len(patient_files)}."

print("Data check passed.")

In [ ]:
def load_set_a(set_a_dir):
    """
    Load all Set A patient files and combine them into
    one long-format DataFrame.
    
    Returns:
        DataFrame with columns:
        RecordID, Time, Parameter, Value
    """

    # Store each patient's data before combining
    all_patients = []

    # Read all patient files in a consistent order
    for file in sorted(set_a_dir.glob("*.txt")):
        df = pd.read_csv(file)

        # Extract the patient's RecordID from the file
        record_id = int(
            df.loc[df["Parameter"] == "RecordID", "Value"].iloc[0]
        )

        # Add RecordID as a separate column
        df["RecordID"] = record_id

        # RecordID is now stored as a column, so remove it
        # from the clinical measurements
        df = df[df["Parameter"] != "RecordID"].copy()

        # Save this patient's data
        all_patients.append(df)

    # Combine all 4000 patient files into one long table
    set_a_long = pd.concat(
        all_patients,
        ignore_index=True
    )

    # Arrange columns in a clear order
    set_a_long = set_a_long[
        ["RecordID", "Time", "Parameter", "Value"]
    ]

    return set_a_long

In [ ]:
# Load and combine all Set A patient records
set_a_long = load_set_a(SET_A_DIR)

# Check the dimensions of the combined data
print("Rows:", len(set_a_long))
print("Patients:", set_a_long["RecordID"].nunique())
print("Variables:", set_a_long["Parameter"].nunique())

# Preview the long-format data
set_a_long.head()

In [ ]:
# Save the combined long-format dataset
set_a_long.to_csv(
    OUT / "set-a_long.csv",
    index=False
)

print("Saved:", OUT / "set-a_long.csv")

In [ ]:
# Create a cleaned copy of the long-format data
clinical_long = set_a_long.copy()

# Treat -1 as missing for variables where -1 is not a valid value
special_missing = (
    clinical_long["Parameter"].isin(["Gender", "Height", "Weight"])
    & (clinical_long["Value"] == -1)
)

clinical_long.loc[special_missing, "Value"] = np.nan

# Convert Time (HH:MM) to hours since ICU admission
time_parts = clinical_long["Time"].str.split(":", expand=True)

clinical_long["Hour"] = (
    time_parts[0].astype(int)
    + time_parts[1].astype(int) / 60
)

# Treat clearly implausible height values as missing
bad_height = (
    (clinical_long["Parameter"] == "Height")
    & (
        (clinical_long["Value"] < 100)
        | (clinical_long["Value"] > 250)
    )
)

print("Implausible height measurements:", bad_height.sum())

clinical_long.loc[bad_height, "Value"] = np.nan

# Check the observation window
print("Minimum hour:", clinical_long["Hour"].min())
print("Maximum hour:", clinical_long["Hour"].max())

In [ ]:
# Load patient outcomes
outcomes = pd.read_csv(OUTCOMES_FILE)

# Keep the original outcomes and create a cleaned copy
outcomes_clean = outcomes.copy()

# In these variables, -1 represents unavailable information
outcome_cols_with_minus1 = [
    "SAPS-I",
    "SOFA",
    "Length_of_stay",
    "Survival"
]

outcomes_clean[outcome_cols_with_minus1] = (
    outcomes_clean[outcome_cols_with_minus1]
    .replace(-1, np.nan)
)

# Confirm that Set A patients match the outcome records
set_a_ids = set(clinical_long["RecordID"].unique())
outcome_ids = set(outcomes_clean["RecordID"].unique())

print("Outcome rows:", len(outcomes_clean))
print("Unique outcome RecordIDs:", outcomes_clean["RecordID"].nunique())
print("Set A IDs without outcome:", len(set_a_ids - outcome_ids))
print("Outcome IDs without Set A record:", len(outcome_ids - set_a_ids))

In [ ]:
# Sort measurements by patient and time so that
# "first" and "last" represent the correct temporal order
clinical_long = clinical_long.sort_values(
    ["RecordID", "Hour"]
)

# Summarize each clinical variable over the 48-hour observation window
wide_summary = (
    clinical_long
    .groupby(["RecordID", "Parameter"])["Value"]
    .agg(["count", "first", "last", "min", "max", "mean"])
)

# Convert the long summaries into one row per patient
wide = wide_summary.unstack("Parameter")

# Flatten the multi-level column names
# Example: ("mean", "HR") becomes "HR_mean"
wide.columns = [
    f"{parameter}_{stat}"
    for stat, parameter in wide.columns
]

wide = wide.reset_index()

print("Wide feature table shape:", wide.shape)
wide.head()

In [ ]:
# Merge patient-level clinical summaries with outcome information
set_a_wide = wide.merge(
    outcomes_clean,
    on="RecordID",
    how="left",
    validate="one_to_one"
)

# Verify that there is exactly one row per patient
print("Final wide table shape:", set_a_wide.shape)
print("Patients:", set_a_wide["RecordID"].nunique())
print("Duplicate RecordIDs:", set_a_wide["RecordID"].duplicated().sum())

# Save the final wide-format dataset
set_a_wide.to_csv(
    OUT / "set-a_wide.csv",
    index=False
)

print("Saved:", OUT / "set-a_wide.csv")

## Information Available at Prediction Time

The goal of this project is to use the information from the first 48 hours after ICU admission to predict in-hospital mortality. Therefore, I should only use information that is already available during these 48 hours.

The patient data include basic information such as age and gender, as well as clinical measurements such as heart rate, blood pressure, and laboratory results. Since many clinical variables are measured multiple times, I summarize them using count, first, last, minimum, maximum, and mean values during the 48-hour period.

Some variables in the outcome file, such as in-hospital death, survival, and length of stay, are known after the prediction period. These variables should not be used as predictors because they contain future information.

In [ ]:
from tableone import TableOne

# Variables included in Table 1
table1_columns = [
    "Age_first",
    "Gender_first",
    "Height_first",
    "Weight_first",
    "ICUType_first",
    "SAPS-I",
    "SOFA"
]

# Variables that should be treated as categorical
categorical = [
    "Gender_first",
    "ICUType_first"
]

# Create Table 1 grouped by in-hospital mortality
table1 = TableOne(
    set_a_wide,
    columns=table1_columns,
    categorical=categorical,
    groupby="In-hospital_death",
    nonnormal=table1_columns,
    missing=True,
    overall=True
)

table1

In [ ]:
# Summarize the primary outcome: in-hospital mortality
outcome_counts = (
    set_a_wide["In-hospital_death"]
    .value_counts()
    .sort_index()
)

outcome_percent = (
    set_a_wide["In-hospital_death"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("Total patients:", len(set_a_wide))
print("Survived:", outcome_counts.get(0, 0))
print("In-hospital deaths:", outcome_counts.get(1, 0))
print(
    "In-hospital mortality:",
    f"{outcome_percent.get(1, 0):.1f}%"
)

outcome_counts

## Outcome Summary

In-hospital death is the main outcome for this prediction task. The code above summarizes the number of patients who survived and died during the hospital stay and shows the overall in-hospital mortality rate.

In [ ]:
# Create a patient-by-variable matrix showing whether
# each variable was ever measured during the 48-hour period

measured_matrix = (
    clinical_long
    .assign(Measured=clinical_long["Value"].notna())
    .groupby(["RecordID", "Parameter"])["Measured"]
    .any()
    .unstack(fill_value=False)
)

# Make sure all 4000 patients are included
measured_matrix = measured_matrix.reindex(
    set_a_wide["RecordID"],
    fill_value=False
)

print("Patients:", measured_matrix.shape[0])
print("Variables:", measured_matrix.shape[1])

measured_matrix.head()

In [ ]:
# Save the measurement matrix for the plotting script
measured_matrix.to_csv(
    OUT / "measurement_matrix.csv"
)

print("Saved:", OUT / "measurement_matrix.csv")

In [ ]:
# Generate the patient-level missingness map
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/plot_missingness.py"],
    check=True
)

print("Missingness map generated.")

In [ ]:
from IPython.display import Image, display

display(
    Image(filename="output/missingness_map.png")
)

### Missingness Map

The patient-level measurement matrix was successfully created to show whether each clinical variable was measured at least once during the first 48 hours.

The visualization step is temporarily skipped because Matplotlib is causing a kernel crash in the current local Jupyter environment. The missingness matrix itself is available for the following analysis.

In [ ]:
# Compare in-hospital mortality between patients who had
# each variable measured and patients who never had it measured

mortality = (
    set_a_wide
    .set_index("RecordID")["In-hospital_death"]
    .reindex(measured_matrix.index)
)

missingness_results = []

for variable in measured_matrix.columns:
    
    # Whether this variable was measured at least once
    measured = measured_matrix[variable]

    # Mortality rate among patients with at least one measurement
    death_rate_measured = mortality[measured].mean() * 100

    # Mortality rate among patients with no measurement
    death_rate_not_measured = mortality[~measured].mean() * 100

    missingness_results.append({
        "Variable": variable,
        "Measured_n": measured.sum(),
        "Never_measured_n": (~measured).sum(),
        "Death_rate_measured": death_rate_measured,
        "Death_rate_never_measured": death_rate_not_measured
    })

missingness_info = pd.DataFrame(missingness_results)

# Calculate the difference in mortality rates
missingness_info["Death_rate_difference"] = (
    missingness_info["Death_rate_measured"]
    - missingness_info["Death_rate_never_measured"]
)

# Sort by the absolute difference in mortality
missingness_info["Absolute_difference"] = (
    missingness_info["Death_rate_difference"].abs()
)

missingness_info = missingness_info.sort_values(
    "Absolute_difference",
    ascending=False
)

missingness_info.head(15)

## Is Missingness Informative?

The results show that missingness may contain useful information about the patients. For several variables, the mortality rate is different between patients who had the variable measured and patients who never had it measured.

For example, patients with measurements for TroponinI, TroponinT, AST, ALT, Bilirubin, and Lactate had higher mortality rates than patients without these measurements. This may happen because some tests are more likely to be ordered for patients with more serious conditions. Therefore, whether a variable was measured may also provide information about patient risk.

Some large differences should be interpreted carefully when the number of patients in one group is very small.

## AI Use Statement

I used ChatGPT as an AI assistant during this assignment. I mainly used it to help me understand Python code, debug errors, organize the data analysis, and explain some statistical and clinical data concepts.

One advantage of using AI was that it helped me find coding problems more quickly and gave me examples of how to organize the analysis. It was also useful when I did not understand the meaning of some Python functions or the purpose of a step in the analysis. However, AI-generated code did not always work directly in my environment, so I needed to test the code and make changes when errors occurred.

I used AI to help review the code, run it step by step, and check the outputs, such as the number of patients, RecordIDs, missing values, variable ranges, and summary statistics. I also needed to compare the results with the structure of the original data to make sure the code was working as expected.

AI also helped me better understand concepts such as long and wide data formats, missing data, patient-level summaries, and why missingness itself may contain information about patient risk.

**Disadvantages.** Sometimes, AI spent too much time focusing on extreme values and made changes without asking me first. AI could not confirm on its own whether the clinical data cleaning cutoffs were appropriate. These experiences showed me that I should check AI-generated code and make sure the analysis met the assignment requirements. I am still responsible for the final code, clinical assumptions, and explanations.